# 04 — Implied Distributions

**Purpose**: For multi-strike markets (daily/weekly/monthly "close above $X"), construct
implied CDFs from the set of strike prices and their probabilities, then compare to
the realized price distribution.

**Outputs**: Distribution comparison plots in `results/thesis_figures/`

In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import interpolate

import config as cfg
import thesis_utils as tu

data_dir, results_dir = tu.ensure_project_dirs(cfg.PROJECT_ROOT)
figures_dir = cfg.FIGURES_DIR
figures_dir.mkdir(parents=True, exist_ok=True)

## 1. Identify Multi-Strike Groups

Multi-strike markets are sets of markets for the same ticker and resolution date with different strike prices.
For example:
- "NVDA closes above $120 on March 18"
- "NVDA closes above $125 on March 18"
- "NVDA closes above $130 on March 18"

Together, these imply a CDF: P(close > $120) = p1, P(close > $125) = p2, etc.

In [ ]:
df = tu.load_parquet('market_resolutions.parquet')
print(f'Resolved markets: {len(df):,}')

# Filter to strike-based markets (not ranges or up/down)
strike_types = {'daily_close_above', 'weekly_above', 'monthly_above'}
df_strike = df[
    df['market_type'].isin(strike_types) &
    df['strike'].notna() &
    df['resolution_date'].notna()
].copy()

df_strike['resolution_date'] = pd.to_datetime(df_strike['resolution_date']).dt.date

print(f'Strike-based markets: {len(df_strike):,}')
print(f'By type: {df_strike["market_type"].value_counts().to_dict()}')

In [ ]:
# Group by (ticker, resolution_date, market_type) to find multi-strike sets
groups = df_strike.groupby(['ticker', 'resolution_date', 'market_type'])
group_sizes = groups.size()

# Multi-strike = groups with 3+ different strikes
multi_strike = group_sizes[group_sizes >= 3]
print(f'Multi-strike groups (3+ strikes): {len(multi_strike)}')
print(f'Total markets in multi-strike groups: {multi_strike.sum()}')

if len(multi_strike) > 0:
    print(f'\nLargest groups:')
    print(multi_strike.sort_values(ascending=False).head(10))

## 2. Construct Implied CDFs

In [ ]:
# Load stock prices for actual close comparison
stock_prices = tu.load_parquet('stock_prices_daily.parquet')
stock_prices.index = pd.to_datetime(stock_prices.index)

implied_cdfs = []

for (ticker, res_date, mtype), group_df in groups:
    if len(group_df) < 3:
        continue

    # Sort by strike price
    g = group_df.sort_values('strike')
    strikes = g['strike'].values

    # If we have implied probabilities, use them; otherwise use outcomes
    if g['implied_prob'].notna().all():
        probs = g['implied_prob'].values
        prob_source = 'implied'
    else:
        # P(close > strike) = 1 - outcome_int gives us the survival function
        # Actually, the market is "will close above $X" so implied_prob = P(close > X)
        # If outcome_int = 1 (yes), the close WAS above X
        probs = g['outcome_int'].values.astype(float)
        prob_source = 'outcome'

    # Survival function: P(close > strike) should be decreasing
    # CDF = 1 - survival
    survival = probs  # P(close > X) = Yes probability
    cdf = 1 - survival

    # Get actual close price
    close_col = f'{ticker}_Close'
    res_ts = pd.Timestamp(res_date)
    actual_close = None
    if close_col in stock_prices.columns:
        valid = stock_prices.index[stock_prices.index <= res_ts]
        if len(valid) > 0:
            actual_close = stock_prices.loc[valid[-1], close_col]

    implied_cdfs.append({
        'ticker': ticker,
        'resolution_date': res_date,
        'market_type': mtype,
        'n_strikes': len(strikes),
        'strikes': strikes,
        'survival': survival,
        'cdf': cdf,
        'actual_close': actual_close,
        'prob_source': prob_source,
    })

print(f'Constructed {len(implied_cdfs)} implied CDFs')

## 3. Plot Implied vs Realized Distributions

In [ ]:
if len(implied_cdfs) > 0:
    # Select a sample of the best CDFs (most strikes) for visualization
    cdf_df = pd.DataFrame(implied_cdfs)
    best = cdf_df.nlargest(min(9, len(cdf_df)), 'n_strikes')

    n_plots = len(best)
    ncols = min(3, n_plots)
    nrows = (n_plots + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 5 * nrows))
    if n_plots == 1:
        axes = [axes]
    else:
        axes = axes.flatten()

    for i, (_, row) in enumerate(best.iterrows()):
        ax = axes[i]
        strikes = row['strikes']
        cdf = row['cdf']
        actual = row['actual_close']

        # Plot implied CDF
        ax.step(strikes, cdf, where='post', color='steelblue', linewidth=2, label='Implied CDF')
        ax.scatter(strikes, cdf, color='steelblue', zorder=5, s=40)

        # Plot actual close
        if actual is not None:
            ax.axvline(actual, color='red', linestyle='--', linewidth=2,
                      label=f'Actual close: ${actual:.2f}')

        ax.set_xlabel('Strike Price ($)')
        ax.set_ylabel('P(Close < Strike)')
        ax.set_title(f'{row["ticker"]} {row["resolution_date"]}\n({row["n_strikes"]} strikes, {row["market_type"]})')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
        ax.set_ylim(-0.05, 1.05)

    for j in range(n_plots, len(axes)):
        axes[j].set_visible(False)

    plt.suptitle('Implied CDFs from Multi-Strike Markets', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.savefig(figures_dir / 'implied_cdfs.png', dpi=cfg.FIGURE_DPI, bbox_inches='tight')
    plt.show()
else:
    print('No multi-strike groups found for CDF construction.')

## 4. Aggregate CDF Accuracy

In [ ]:
if len(implied_cdfs) > 0:
    # For each implied CDF, compute where the actual close falls in the distribution
    # This gives us a PIT (Probability Integral Transform) value
    # If CDFs are well-calibrated, PIT values should be uniform
    pit_values = []
    for entry in implied_cdfs:
        actual = entry['actual_close']
        if actual is None:
            continue
        strikes = entry['strikes']
        cdf = entry['cdf']

        # Interpolate CDF at actual close price
        if actual <= strikes[0]:
            pit = cdf[0]
        elif actual >= strikes[-1]:
            pit = cdf[-1]
        else:
            f_interp = interpolate.interp1d(strikes, cdf, kind='linear', fill_value='extrapolate')
            pit = float(f_interp(actual))

        pit = np.clip(pit, 0, 1)
        pit_values.append(pit)

    if pit_values:
        pit_arr = np.array(pit_values)
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))

        # PIT histogram — should be uniform if well-calibrated
        axes[0].hist(pit_arr, bins=10, range=(0, 1), edgecolor='black', color='steelblue', alpha=0.7)
        axes[0].axhline(len(pit_arr) / 10, color='red', linestyle='--', label='Uniform expectation')
        axes[0].set_xlabel('PIT Value')
        axes[0].set_ylabel('Count')
        axes[0].set_title(f'PIT Histogram (N={len(pit_arr)})')
        axes[0].legend()

        # PIT QQ plot
        pit_sorted = np.sort(pit_arr)
        expected = np.linspace(0, 1, len(pit_sorted))
        axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.5)
        axes[1].plot(expected, pit_sorted, 'o-', color='steelblue', markersize=3)
        axes[1].set_xlabel('Expected (Uniform)')
        axes[1].set_ylabel('Observed PIT')
        axes[1].set_title('PIT Q-Q Plot')

        plt.tight_layout()
        plt.savefig(figures_dir / 'pit_analysis.png', dpi=cfg.FIGURE_DPI, bbox_inches='tight')
        plt.show()

        from scipy import stats
        ks_stat, ks_p = stats.kstest(pit_arr, 'uniform')
        print(f'KS test for uniformity: stat={ks_stat:.4f}, p={ks_p:.4f}')
        print(f'  {"Reject" if ks_p < 0.05 else "Fail to reject"} uniformity at 5% level')